# `Chroma DB`

In [26]:
# !pip install chromadb

In [27]:
import chromadb

client = chromadb.PersistentClient(path = '/content/chromdb')

In [28]:
# Create a collection to store vectors
collection = client.create_collection(name="collection7")

# Insert sample data (ID, embeddings, metadata)
collection.add(
    ids=["1", "2", "3"],
    embeddings=[[0.1, 0.2, 0.3], [0.2, 0.3, 0.4], [0.3, 0.4, 0.5]],
    metadatas=[{"name": "Keerti"}, {"name": "Amit"}, {"name": "Educosys"}]
)

In [29]:
# TODO: List of collections
print("Available Collections:", client.list_collections())

Available Collections: [Collection(name=collection1), Collection(name=collection4), Collection(name=collection5), Collection(name=collection7), Collection(name=collection3), Collection(name=collection6), Collection(name=collection2)]


In [30]:
# This will not give the embedding coz of the design of vector DB
print("Fetching data with ID 1:", collection.get(ids=["1"]))

Fetching data with ID 1: {'ids': ['1'], 'embeddings': None, 'documents': [None], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'name': 'Keerti'}]}


In [31]:
# This will give the embedding also
print(
    "Fetching data with ID 1:",
    collection.get(ids=["4"], include=["embeddings", "metadatas"])
)

Fetching data with ID 1: {'ids': [], 'embeddings': array([], dtype=float64), 'documents': None, 'uris': None, 'included': ['embeddings', 'metadatas'], 'data': None, 'metadatas': []}


In [32]:
#
collection.add(
    ids = ['4'],
    embeddings = [[0.1,0.2,0.3]],
    documents = ["Someone is a software engineer with 5 years of exp."]
)

In [33]:
# This will not give the embedding coz of the design of vector DB
print("Fetching data with ID 1:", collection.get(ids=["4"]))

Fetching data with ID 1: {'ids': ['4'], 'embeddings': None, 'documents': ['Someone is a software engineer with 5 years of exp.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None]}


In [34]:
collection.query(
    query_embeddings = [[0.3, 0.4, 0.5]],
    n_results = 1,
    include = ['documents']
)

{'ids': [['3']],
 'embeddings': None,
 'documents': [[None]],
 'uris': None,
 'included': ['documents'],
 'data': None,
 'metadatas': None,
 'distances': None}

In [35]:
# This will give the embedding also
print(
    "Fetching data with ID 1:",
    collection.get(ids=["2"], include=["embeddings", "metadatas"])
)

Fetching data with ID 1: {'ids': ['2'], 'embeddings': array([[0.2       , 0.30000001, 0.40000001]]), 'documents': None, 'uris': None, 'included': ['embeddings', 'metadatas'], 'data': None, 'metadatas': [{'name': 'Amit'}]}


In [36]:
# Updating an existing entry
collection.update(
    ids = ['2'],
    embeddings = [[0.5,0.5,0.5]],
    metadatas = [{'name':'Bob Updated'}]
)
print("Updated Entry:",collection.get(ids = ['2']))

Updated Entry: {'ids': ['2'], 'embeddings': None, 'documents': [None], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'name': 'Bob Updated'}]}


In [37]:
# This will not give the embedding coz of the design of vector DB
print("Fetching data with ID 1:", collection.get(ids=["2"]))

Fetching data with ID 1: {'ids': ['2'], 'embeddings': None, 'documents': [None], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'name': 'Bob Updated'}]}


In [38]:
# Deleting an existing entry
collection.delete(ids = ['2'],)
print("Updated Entry:",collection.get(ids = ['2']))

Updated Entry: {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


# `OpenAI Embeddings`

In [39]:
import os
import chromadb

# Set your OpenAI API key directly from environment or .env file
os.environ['EURI_API_TOKEN'] = os.getenv('EURI_API_TOKEN', 'your-openai-key-here')

client = chromadb.PersistentClient(path="./chroma_db")  # This persists data

# Reset collection to ensure dimension match
try:
    client.delete_collection(name="collection_euri_final_v1")
    print("Collection deleted to reset dimensions.")
except Exception as e:
    print("Collection did not exist or could not be deleted:", e)

collection = client.get_or_create_collection(name="collection_euri_final_v1")
print("Collection created/loaded: collection_euri_final_v1")

Collection did not exist or could not be deleted: Collection [collection_euri_final_v1] does not exist
Collection created/loaded: collection_euri_final_v1


In [40]:
import openai
import os

# Configure OpenAI to use Euri API
openai.api_key = os.getenv('EURI_API_TOKEN')
openai.base_url = 'https://api.euron.one/api/v1/euri/'  # Euri API base URL for embeddings

def get_openai_embedding(text):
    response = openai.embeddings.create(
        input=[text],
        model="text-embedding-ada-002"  # Euri supports this model name
    )
    return response.data[0].embedding


In [41]:
documents = [
    "The Eiffel Tower is located in Paris.",
    "The Colosseum is in Rome, Italy.",
    "The Taj Mahal is a famous monument in India.",
    "Mount Everest is the highest mountain in the world.",
    "Python is a popular programming language."
]

# Convert documents to embeddings using Euri API
embeddings = [get_openai_embedding(doc) for doc in documents]

# Insert into ChromaDB
collection.add(
    ids=[str(i) for i in range(len(documents))],  # Unique IDs
    documents=documents,
    embeddings=embeddings
)

print("Data added successfully! ??")

Data added successfully! ??


In [56]:
tower_ht_text = "Eiffel tower is 300 tall."

collection.add(
    ids = ['6'],
    embeddings = get_openai_embedding(tower_ht_text),
    metadatas = [{"text": tower_ht_text}]
)

In [58]:
query_text = "Whats the height of Eiffel Tower?"
query_embedding = get_openai_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2, # Get top 2 matches
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])

Query: Whats the height of Eiffel Tower?
Most Similar Result: [None, 'The Eiffel Tower is located in Paris.']
Distance: [0.18159973621368408, 0.2560642957687378]


# Adding new document (Height info)

In [ ]:
tower_ht_text = "Eiffel tower is 300 tall."

# Fix: Wrap embedding in a list and include 'documents' so it can be retrieved
collection.add(
    ids=['6'],
    documents=[tower_ht_text],  # Added this!
    embeddings=[get_openai_embedding(tower_ht_text)], # Wrapped in list!
    metadatas=[{"text": tower_ht_text}]
)
print("Added document: '" + tower_ht_text + "'")

In [ ]:
# Query for the new information
query_text = "How tall is the Eiffel Tower?"
query_embedding = get_openai_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=1,
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])